# 🔮 HyDE — Hypothetical Document Embeddings

**HyDE** (Gao et al., [arXiv:2212.10496](https://arxiv.org/abs/2212.10496)) is a query-transformation
technique for RAG. Instead of embedding the user's question and searching with it, you first ask an
LLM to *invent an answer*, then search using the embedding of that invented answer.

The invented answer is usually wrong in its details — that's fine. It is not shown to the user and
never enters the final context. It exists only to produce a **better search vector**.

## Learning Objectives
1. **The asymmetry problem** — why a short question and a long answer land far apart in embedding space
2. **Manual HyDE** — build the generate → embed → search loop by hand and inspect every intermediate
3. **Packaged HyDE** — use `HypotheticalDocumentEmbedder` and understand its asymmetric design
4. **Wiring HyDE into retrieval** — the two ways to actually connect it to a vector store
5. **Evaluating the tradeoff** — compare against a plain baseline and know when HyDE hurts

## Prerequisites
- A `.env` at the repo root with `OPENAI_API_KEY` and `EXPERIENTIALLABS_API_KEY`
- Source documents in `04_Retrieval_and_RAG/shared_data/`
- Familiarity with embeddings and vector stores (see `01_Introduction_to_RAG/`)

---
## 🧠 Part 1: The Problem HyDE Solves

Dense retrieval works by embedding the query and the documents into the same vector space and taking
whatever is nearest. That quietly assumes questions and answers *look alike* to the embedding model.
They often don't.

Compare what you actually search with against what you hope to find:

| | Text | Shape |
|---|---|---|
| **Query** | *"What is LangSmith, and why do we need it?"* | 9 words, interrogative, no domain vocabulary |
| **Target chunk** | *"LangSmith gives you full visibility into model inputs and output of every step in the chain of events. This makes it easy for teams to experiment with new chains and prompt templates…"* | 400 tokens, declarative, dense with terminology |

These are different genres of text. The embedding sits closer to *other questions* than to the
passage that answers it — the **query–document asymmetry** problem.

### The HyDE Insight

> Don't search with the question. Search with a **fake answer** to the question.

An LLM-written passage is declarative, long, and full of the same vocabulary real documentation uses.
It is the same *genre* as the target chunk, so it lands much closer to it in vector space — even when
its facts are hallucinated.

**Three steps, always:**

1. **Generate** — LLM writes a hypothetical answer to the question
2. **Embed** — turn that hypothetical answer into a vector
3. **Search** — retrieve real chunks nearest to that vector

The retrieved chunks are real. Only the search key was imaginary.

---
## ⚙️ Part 2: Environment Setup

Standard setup: imports, credentials, and the two models HyDE requires.

### 2.1 Imports

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports
# ============================================================================
import os
import warnings

import numpy as np
from dotenv import load_dotenv

# LangChain core building blocks
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Integrations
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# The packaged HyDE implementation and its built-in prompt library
from langchain_classic.chains import HypotheticalDocumentEmbedder
from langchain_classic.chains.hyde.prompts import PROMPT_MAP

# Project helper (LLM factory)
from helpers import get_experientiallabs_llm

warnings.filterwarnings("ignore")

print("✅ Imports loaded successfully!")

### 2.2 Credentials and LangSmith Tracing

`load_dotenv()` reads the repo-root `.env`. Setting `LANGSMITH_PROJECT` groups every run from this
notebook under its own project, so the HyDE traces don't mix with other notebooks'.

> **Note**: tracing only activates if `.env` also sets `LANGSMITH_TRACING=true` (or the legacy
> `LANGCHAIN_TRACING_V2=true`). A misspelled variable silently disables tracing with no error.

In [ ]:
# ============================================================================
# CONFIGURATION: Credentials and tracing
# ============================================================================
load_dotenv()

os.environ["LANGSMITH_PROJECT"] = "HyDE"

print(f"✅ OpenAI key present:  {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"✅ LangSmith tracing:   {os.getenv('LANGSMITH_TRACING')}")
print(f"✅ LangSmith project:   {os.environ['LANGSMITH_PROJECT']}")

### 2.3 Initialize the Two Models

**HyDE always needs two models**, doing two different jobs. This trips people up, so it is worth
stating plainly:

| Model | Role in HyDE |
|---|---|
| **LLM** (`get_experientiallabs_llm()`) | Writes the hypothetical answer — the *generate* step |
| **Embedding model** (`OpenAIEmbeddings()`) | Turns text into vectors — the *embed* step |

The LLM never sees your documents and the embedding model never writes text. Neither can do HyDE alone.

In [ ]:
# ============================================================================
# MODEL INITIALIZATION: One LLM (generates) + one embedder (vectorizes)
# ============================================================================
llm = get_experientiallabs_llm()

# Pinned explicitly: bare OpenAIEmbeddings() still defaults to the legacy
# text-embedding-ada-002. What matters most is that the SAME model embeds both
# the documents and the hypothetical answer — otherwise the vectors aren't comparable.
base_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"🤖 LLM:        {llm.model_name}")
print(f"🔢 Embeddings: {base_embeddings.model}")

---
## 📚 Part 3: Build the Knowledge Base

Two LangChain blog posts, chunked and indexed. Nothing HyDE-specific happens here — this is an
ordinary RAG index, which is exactly the point of the next section's warning.

### 3.1 Load and Split

In [ ]:
# ============================================================================
# KNOWLEDGE BASE: Load source documents and split into chunks
# ============================================================================
loaders = [
    TextLoader("../../shared_data/blog.langchain.dev_announcing-langsmith_.txt", encoding="utf-8"),
    TextLoader("../../shared_data/blog.langchain.dev_automating-web-research_.txt", encoding="utf-8"),
]

docs = []
for loader in loaders:
    docs.extend(loader.load())

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=400,    # measured in tokens, not characters
    chunk_overlap=60,  # carry-over so ideas aren't severed at a boundary
)
splits = text_splitter.split_documents(docs)

print(f"📄 Loaded {len(docs)} documents → split into {len(splits)} chunks")

### 3.2 Index into Chroma

> **Key Insight**: the index is built with the **plain** embedding model, not the HyDE one.
> HyDE changes how the **query** is embedded — never how documents are stored. Documents are already
> answer-shaped; they need no transformation. Keeping indexing plain also means you don't pay for an
> LLM call per chunk.

In [ ]:
# ============================================================================
# VECTOR STORE: Index chunks with the PLAIN embedder
# ============================================================================
vectorstore = Chroma.from_documents(documents=splits, embedding=base_embeddings)

print(f"✅ Indexed {len(splits)} chunks into Chroma")

---
## 🔍 Part 4: Baseline — Retrieval Without HyDE

Before adding a technique, measure what you have without it. This is a plain similarity search using
the raw question, and it is the yardstick every later result is compared against.

In [ ]:
# ============================================================================
# BASELINE: Retrieve using the raw question
# ============================================================================
question = "What is LangSmith, and why do we need it?"

plain_docs = vectorstore.similarity_search(question, k=4)

print(f"🔍 Baseline results for: {question!r}\n")
for i, doc in enumerate(plain_docs, 1):
    print(f"[{i}] {doc.page_content[:180].strip()}...\n")

---
## ✍️ Part 5: HyDE Step by Step (Manual)

Build the three steps by hand first. Doing it manually is the whole lesson — you get to *see* the
hypothetical document, which the packaged class hides inside `embed_query()`.

### 5.1 Step 1 — A Chain That Invents an Answer

The instruction matters: asking for *"a passage as if from documentation"* produces text in the same
genre as the indexed chunks. Asking for *"a helpful reply"* would produce chatty prose that embeds
further away from the target.

In [ ]:
# ============================================================================
# HYDE STEP 1: Chain that writes a hypothetical answer
# ============================================================================
system = """You are a knowledgeable research assistant.
Write a short, factual passage that answers the user's question.
Write it as an excerpt from technical documentation or an engineering blog post."""

hyde_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

generate_hypothetical_doc = hyde_prompt | llm | StrOutputParser()

print("✅ Hypothetical-document chain ready")

### 5.2 Step 2 — Generate and Inspect the Hypothetical Document

This is the cell worth staring at. Read the output and note two things:

- it is **declarative and documentation-shaped**, unlike the question
- some of its specifics may be **wrong or invented** — and that does not matter, because this text is
  thrown away after embedding. It is a search key, not an answer.

In [ ]:
# ============================================================================
# HYDE STEP 2: Generate the hypothetical document and look at it
# ============================================================================
hypothetical_doc = generate_hypothetical_doc.invoke({"question": question})

print("📄 Hypothetical document — invented by the LLM, never shown to the user:\n")
print(hypothetical_doc)
print(f"\n📋 Question length:   {len(question.split()):>4} words")
print(f"📋 Hypothetical doc:  {len(hypothetical_doc.split()):>4} words")

### 5.3 Step 3 — Embed the Hypothetical Document and Search With It

`similarity_search_by_vector()` is the method that makes this work: it lets you supply the search
vector directly, instead of handing the store a string for it to embed itself.

In [ ]:
# ============================================================================
# HYDE STEP 3: Embed the hypothetical answer, search with THAT vector
# ============================================================================
hyde_vector = base_embeddings.embed_query(hypothetical_doc)
manual_hyde_docs = vectorstore.similarity_search_by_vector(hyde_vector, k=4)

print(f"🔍 HyDE results (searched with the hypothetical answer, not the question)\n")
for i, doc in enumerate(manual_hyde_docs, 1):
    print(f"[{i}] {doc.page_content[:180].strip()}...\n")

---
## 📦 Part 6: HyDE the Packaged Way

`HypotheticalDocumentEmbedder` bundles the same three steps behind the standard `Embeddings`
interface, so it can be dropped anywhere an embedding model is expected.

### 6.1 Build the Embedder

It takes the two models from Part 2.3 plus a **prompt key** naming one of eight built-in templates
(inherited from the BEIR benchmark datasets the HyDE paper evaluated on). `web_search` is the
general-purpose one; its template is literally `"Please write a passage to answer the question"`.

In [ ]:
# ============================================================================
# PACKAGED HYDE: Build the embedder
# ============================================================================
hyde_embeddings = HypotheticalDocumentEmbedder.from_llm(
    llm,              # generates the hypothetical answer
    base_embeddings,  # embeds it
    "web_search",     # built-in prompt template key
)

print("✅ HyDE embedder built with prompt key 'web_search'")
print(f"📋 Available prompt keys: {list(PROMPT_MAP.keys())}")

### 6.2 The Key Asymmetry

This is the design detail that makes the class usable, and it is easy to miss:

| Method | Behavior |
|---|---|
| `embed_documents()` | **Pass-through.** Delegates straight to the base embedder — no LLM call |
| `embed_query()` | **HyDE.** Runs the LLM first, embeds the generated text |

Documents are indexed normally; only queries pay the extra LLM call. The cell below proves it by
checking that a document embedded through the HyDE wrapper is byte-identical to one embedded plainly.

In [ ]:
# ============================================================================
# ASYMMETRY CHECK: embed_documents passes through, embed_query does not
# ============================================================================
sample = "LangSmith gives full visibility into every step of an LLM chain."

via_hyde = hyde_embeddings.embed_documents([sample])[0]
via_plain = base_embeddings.embed_documents([sample])[0]

print(f"📋 embed_documents identical to plain embedder? {np.allclose(via_hyde, via_plain)}")
print("   → indexing costs no LLM calls\n")

q_vec = hyde_embeddings.embed_query(question)  # this one DOES call the LLM
print(f"📋 embed_query returned a {len(q_vec)}-dim vector (after an LLM round-trip)")

### 6.3 Wiring Option A — Embed the Query, Search by Vector

Reuses the existing plain index. Both sides use the same underlying embedding model, so the vectors
live in the same space and are directly comparable.

In [ ]:
# ============================================================================
# WIRING (A): embed_query → similarity_search_by_vector
# ============================================================================
packaged_hyde_docs = vectorstore.similarity_search_by_vector(q_vec, k=4)

print("🔍 Packaged HyDE results\n")
for i, doc in enumerate(packaged_hyde_docs, 1):
    print(f"[{i}] {doc.page_content[:180].strip()}...\n")

### 6.4 Wiring Option B — Hand the Embedder to the Store

Because `embed_documents()` is a pass-through, you can pass the HyDE embedder straight to
`Chroma.from_documents()`. Indexing stays plain and cheap; every subsequent `similarity_search()`
transparently applies HyDE. This is the drop-in form — no call-site changes anywhere else.

In [ ]:
# ============================================================================
# WIRING (B): HyDE embedder as the store's embedding function
# ============================================================================
hyde_store = Chroma.from_documents(documents=splits, embedding=hyde_embeddings)

dropin_docs = hyde_store.similarity_search(question, k=4)  # HyDE applied automatically

print("🔍 Drop-in HyDE results\n")
for i, doc in enumerate(dropin_docs, 1):
    print(f"[{i}] {doc.page_content[:180].strip()}...\n")

---
## ⚖️ Part 7: Does It Actually Help?

Put the baseline and the HyDE results side by side. On a small, well-matched corpus like this one the
two often overlap heavily — which is itself the lesson: **HyDE is not free, and it is not always a win.**

In [ ]:
# ============================================================================
# COMPARISON: Plain retrieval vs HyDE retrieval
# ============================================================================
def preview(doc, n=70):
    return doc.page_content[:n].strip().replace("\n", " ")

print(f"{'rank':<6}{'PLAIN (raw question)':<75}{'HyDE (hypothetical answer)'}")
print("-" * 150)
for i, (p, h) in enumerate(zip(plain_docs, packaged_hyde_docs), 1):
    print(f"{i:<6}{preview(p):<75}{preview(h)}")

overlap = len({p.page_content for p in plain_docs} & {h.page_content for h in packaged_hyde_docs})
print(f"\n📊 Chunks retrieved by BOTH methods: {overlap} / {len(plain_docs)}")

### The Tradeoff

| | Plain retrieval | HyDE retrieval |
|---|---|---|
| **Latency** | One embedding call | One **LLM** call + one embedding call |
| **Cost per query** | Cents per thousand | An LLM generation every single query |
| **Short/vague queries** | Often misses | Clear win — the LLM supplies missing vocabulary |
| **Niche or private domains** | Retrieves what's there | ⚠️ Risky — the LLM invents plausible-but-wrong jargon and steers *away* from the right chunks |
| **Keyword-ish queries** | Strong | Often no better, sometimes worse |

**Use HyDE when** queries are short, underspecified, or phrased very differently from your documents,
and the subject matter is well represented in the LLM's training data.

**Avoid HyDE when** your corpus is proprietary or highly specialized (the model has nothing real to
hallucinate from), when latency budgets are tight, or when queries already read like documents.

---
## 🎛️ Part 8: Multi-Document HyDE

The original paper generates **several** hypothetical answers and averages their embeddings. Averaging
cancels out the idiosyncrasies of any single hallucination, giving a more stable search vector.

> **⚠️ Gotcha**: `HypotheticalDocumentEmbedder` can no longer do this itself. `from_llm()` builds
> `prompt | llm | StrOutputParser()`, whose output is one string, and `embed_query()` wraps it in a
> one-element list — so `combine_embeddings()` always averages exactly one vector. The older
> `LLMChain`-based implementation supported `n=4`; the Runnable-based one does not. Passing a
> differently-named LLM changes nothing. To average N hypotheses, generate them yourself.

In [ ]:
# ============================================================================
# MULTI-DOCUMENT HYDE: Average several hypothetical answers
# ============================================================================
N = 3
varied_llm = get_experientiallabs_llm(temperature=0.8)  # variety requires temperature > 0
varied_chain = hyde_prompt | varied_llm | StrOutputParser()

hypotheticals = varied_chain.batch([{"question": question}] * N)

vectors = base_embeddings.embed_documents(hypotheticals)
averaged_vector = np.mean(np.array(vectors), axis=0).tolist()

multi_hyde_docs = vectorstore.similarity_search_by_vector(averaged_vector, k=4)

print(f"✅ Averaged {N} hypothetical documents into one query vector\n")
for i, doc in enumerate(multi_hyde_docs, 1):
    print(f"🔍 [{i}] {doc.page_content[:180].strip()}...\n")

### Inspect the Variation

Worth confirming the N hypotheticals actually differ — at `temperature=0` they would be near-identical
and averaging would buy you nothing but three times the cost.

In [ ]:
# ============================================================================
# VARIATION CHECK: How different were the hypothetical documents?
# ============================================================================
for i, doc in enumerate(hypotheticals, 1):
    print(f"📄 [{i}] {doc[:130].strip()}...\n")

sims = [
    float(np.dot(vectors[0], v) / (np.linalg.norm(vectors[0]) * np.linalg.norm(v)))
    for v in vectors[1:]
]
print(f"📊 Cosine similarity of #1 against the others: {[round(s, 4) for s in sims]}")
print("   → close to 1.0 means little variation, so averaging adds little")

---
## 📝 Summary

### 1. The Problem
- Dense retrieval assumes queries and documents look alike; short questions and long passages sit far
  apart in embedding space — the **query–document asymmetry**.

### 2. The HyDE Technique
- **Generate → Embed → Search**: an LLM invents an answer, you embed the fake answer, and retrieve
  real chunks nearest to it.
- **Factual accuracy of the hypothetical document is irrelevant.** It is a search key, discarded after
  embedding, and never reaches the user.

### 3. Two Implementations
- **Manual** (Part 5) — you see the hypothetical document; best for learning and for controlling the
  generation prompt.
- **Packaged** (Part 6) — `HypotheticalDocumentEmbedder` behind the standard `Embeddings` interface.

### 4. The Asymmetry That Makes It Practical
- `embed_documents()` passes through to the base embedder; `embed_query()` runs the LLM.
- Indexing stays cheap; only queries pay for generation. This is why the embedder can be dropped
  straight into `Chroma.from_documents()`.

### 5. Wiring It Up
- **Option A**: `embed_query()` → `similarity_search_by_vector()` — explicit, reuses a plain index.
- **Option B**: pass the HyDE embedder as the store's embedding function — transparent drop-in.

### 6. Knowing When Not To Use It
- Costs an LLM call per query. Wins on short/vague queries in well-known domains; can actively hurt on
  proprietary corpora where the model hallucinates misleading vocabulary.
- **Always compare against a plain baseline** (Part 7) rather than assuming it helps.

### 7. Multi-Document HyDE
- Averaging several hypotheticals stabilizes the search vector, but must be hand-rolled — the packaged
  class collapses to a single document on current LangChain versions.

### Next Steps
- Inspect these runs in LangSmith under the **HyDE** project to see the extra LLM call HyDE inserts
  ahead of every retrieval.
- Compare with the sibling techniques in this folder: `a. Multi_Query`, `b. RAG_Fusion`,
  `c. Step_Back_Prompting` — all four rewrite the query, but each in a different direction.